## Setup
- no GPU needed to run this notebook.

### Build the project from its repo

#### clone the repo
```bash
git clone https://github.com/ggml-org/llama.cpp
cd llama.cpp
```

#### install cmake
```bash
sudo apt update
sudo apt install -y build-essential cmake pkg-config
```
#### verify cmake installation
```bash
g++ --version
cmake --version
```
#### isntall ninja for faster build
```bash
sudo apt install -y ninja-build
sudo apt install -y libssl-dev pkg-config
```

#### build the project using cmake and ninja
```bash
rm -rf build
cmake -B build -G Ninja -DCMAKE_BUILD_TYPE=Release
cmake --build build
```
#### reset the pwd to the project root dir
```bash
cd ..
```

### Run the server
- if you want to use llama-cli to use the model (explained below), you can skip this step 

#### using HF repo
```bash
./llama.cpp/build/bin/llama-server \
    -hf ggml-org/gemma-3-1b-it-GGUF \
    --host localhost \
    --port 8080 \
    -c 1000 \
    --threads 4 \           # CPU threads
    --n-gpu-layers 0 \
    --mlock \               # Lock memory to prevent swapping
    --cont-batching         # Enable continuous batching
```

#### using a local model (GGUF file)
```bash
hf download HuggingFaceTB/SmolLM2-1.7B-Instruct-GGUF smollm2-1.7b-instruct-q4_k_m.gguf \
        --local-dir . \
        --local-dir-use-symlinks False
```
and then
```bash
./llama.cpp/build/bin/llama-server \
    -m ./smollm2-1.7b-instruct-q4_k_m.gguf \
    --host localhost \
    --port 8080 \
    -c 1000 \
    --n-gpu-layers 0
```

## Inference

### Uisng `llama-cli`
- in this case, we don't need to run the server (explained above).
- we directly request the llama-cli from terminal
- and it returns a single chat compeltion response 

#### using HF repo model
```bash
./llama.cpp/build/bin/llama-cli -hf ggml-org/gemma-3-1b-it-GGUF -p "who am i???"
```

#### using a local model
- first, download the model
```bash
hf download HuggingFaceTB/SmolLM2-1.7B-Instruct-GGUF smollm2-1.7b-instruct-q4_k_m.gguf \
        --local-dir . \
        --local-dir-use-symlinks False
```
- then
```bash
./llama.cpp/build/bin/llama-cli -m ./smollm2-1.7b-instruct-q4_k_m.gguf -p "who am i???"

```

### Using python SDK

In [ ]:
# ! conda install "huggingface_hub[cli]"
# ! pip install openai
# ! pip install llama-cpp-python

In [ ]:
from huggingface_hub import InferenceClient
from openai import OpenAI
from llama_cpp import Llama

In [ ]:
endpoint = "http://127.0.0.1:8080"
messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a creative story about space exploration"},
    ]

#### HF Inference Client

In [ ]:
client = InferenceClient(model=endpoint)

response = client.chat_completion(
    messages=messages,
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
    frequency_penalty=1.0,  # Example value: discourages frequent repetition
    presence_penalty=0.5,   # Example value: encourages new topics/tokens
)
print(response.choices[0].message.content)

#### OpenAI Inference Client

In [ ]:
client = OpenAI(base_url=endpoint, api_key="")

response = client.chat.completions.create(
    model="smollm2-1.7b-instruct",
    # Model identifier can be anything as server only loads one model
    messages=messages,
    max_tokens=100,
    temperature=0.7,
    top_p=0.95,
    # stop=["\n\n", "###"] # stops generating when sees these tokens
)
print(response.choices[0].message.content)

#### Llama native python library

In [ ]:
llm = Llama(
    model_path="./smollm2-1.7b-instruct-q4_k_m.gguf",
    n_ctx=4096,  # Context window size
    n_threads=8,  # CPU threads
    n_gpu_layers=0,  # GPU layers (0 = CPU only)
)

# Format prompt according to the model's expected format
prompt = """<|im_start|>system
You are a creative storyteller.
<|im_end|>
<|im_start|>user
Write a creative story
<|im_end|>
<|im_start|>assistant
"""

# Generate response with precise parameter control
output = llm(
    prompt,
    max_tokens=200,
    temperature=0.8,
    top_p=0.95,
    frequency_penalty=0.5,
    presence_penalty=0.5,
    repeat_penalty=1.1,
    stop=["<|im_end|>"],
)
# frequency_penalty: The penalty to apply to tokens based on their frequency in the prompt.
# presence_penalty: The penalty to apply to tokens based on their presence in the prompt.
# repeat_penalty: The penalty to apply to repeated tokens.

print(output["choices"][0]["text"])